In [ ]:
!nvidia-smi

In [ ]:
%cd /content/brisc-xai
from google.colab import files
files.upload()

In [ ]:
!ls -la /content/brisc-xai
!pip install -q -r /content/brisc-xai/requirements.txt
%cd /content/brisc-xai
!python 01_build_dataset.py

In [ ]:
%cd /content/brisc-xai
from google.colab import files
files.upload()      # config.py et 02_train_ablation.py
!python 02_train_ablation.py

In [ ]:
!cp -r /content/brisc-xai /content/drive/MyDrive/

In [ ]:
!cp -r /content/brisc-xai /content/drive/MyDrive/

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
P = "/content/drive/MyDrive/classification_task/repro"
print("runs.csv :", os.path.exists(f"{P}/ablation/ablation_runs.csv"))
print("models   :", len(os.listdir(f"{P}/ablation/models")))

In [ ]:
import os, getpass
!pip install -q kaggle
if not os.path.isdir("/content/brisc_fresh/brisc2025"):
    tok = getpass.getpass("token kaggle : ")
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    open(os.path.expanduser("~/.kaggle/access_token"), "w").write(tok.strip())
    os.chmod(os.path.expanduser("~/.kaggle/access_token"), 0o600)
    del tok
    !kaggle datasets download -d briscdataset/brisc2025 -p /content/brisc_fresh --unzip

os.environ["BRISC_ROOT"] = "/content/brisc_fresh/brisc2025"
os.environ["BRISC_OUT"]  = "/content/drive/MyDrive/classification_task/repro"
os.makedirs("/content/brisc-xai", exist_ok=True)
print(os.path.isdir(os.environ["BRISC_ROOT"]))

In [ ]:
%cd /content/brisc-xai
from google.colab import files
files.upload()

In [ ]:
!python 03_measure_xai.py

In [ ]:
import os, sys, numpy as np, pandas as pd, cv2, tensorflow as tf
sys.path.insert(0, "/content/brisc-xai")
import config as C
from importlib.machinery import SourceFileLoader
m3 = SourceFileLoader("m3", "/content/brisc-xai/03_measure_xai.py").load_module()

S = C.PROTOCOL["img_size"]
mani = pd.read_csv(os.path.join(C.DATASET_DIR, "manifest.csv"))
xai = mani[(mani.split == "test") & mani.mask_path.notna()
           & mani.label.isin(C.TUMOR_CLASSES)].reset_index(drop=True)
IMGS = np.stack([tf.keras.utils.img_to_array(
    tf.keras.utils.load_img(p, target_size=(S, S))).astype("float32") / 255.0
    for p in xai.path])
BRAIN = np.stack([m3.brain_mask(im) for im in IMGS])

f = BRAIN.reshape(len(BRAIN), -1).mean(axis=1)
print(f"min {f.min():.3f} | sous 0.15 : {(f < 0.15).sum()} images sur {len(f)}")
print(pd.Series(f).describe().round(3).to_string())
print(xai.loc[f < 0.15, ["stem", "label"]].to_string())

In [ ]:
%cd /content/brisc-xai
from google.colab import files
files.upload()      # 03_measure_xai.py
!python 03_measure_xai.py

In [ ]:
%cd /content/brisc-xai
!rm -f "03_measure_xai.py" "03_measure_xai (1).py" "03_measure_xai (2).py" "03_measure_xai (3).py"
from google.colab import files
files.upload()      # 03_measure_xai.py
!ls -la /content/brisc-xai

In [ ]:
%cd /content/brisc-xai
!rm -rf __pycache__ *\(*\)*
!ls -la

In [ ]:
!grep -c "NOTE:" 03_measure_xai.py
!python 03_measure_xai.py

In [ ]:
!python 04_make_tables.py

In [ ]:
!cp -r /content/drive/MyDrive/classification_task/repro /content/drive/MyDrive/classification_task/repro_3seeds
!cp -r /content/brisc-xai /content/drive/MyDrive/

In [ ]:
%cd /content/brisc-xai
!mv "config (1).py" config.py
!ls -la config.py
!python -c "import config as C; print(C.PROTOCOL['epochs'], C.PROTOCOL['seeds'], C.PROTOCOL_HASH)"

In [ ]:
P = "/content/drive/MyDrive/classification_task/repro"
!rm -f "{P}/ablation/ablation_runs.csv"
!rm -rf "{P}/ablation/models" "{P}/ablation/histories"
!python 02_train_ablation.py

In [ ]:
!python 03_measure_xai.py
!python 04_make_tables.py

In [ ]:
!python 04_make_tables.py

In [ ]:
!cp -r /content/drive/MyDrive/classification_task/repro /content/drive/MyDrive/classification_task/repro_5seeds_final

In [ ]:
import pandas as pd
r = pd.read_csv("/content/drive/MyDrive/classification_task/repro/ablation/ablation_runs.csv")
print(r.shape, r.protocol_hash.unique())
print(r[["arch","seed","accuracy","best_epoch","hit_cap"]].sort_values(["arch","seed"]).to_string(index=False))
print(r.groupby("arch").accuracy.agg(["count","mean","std"]).round(4).to_string())

In [ ]:
import pandas as pd
r = pd.read_csv("/content/drive/MyDrive/classification_task/repro/ablation/ablation_runs.csv")
print(r.shape, r.protocol_hash.unique())
print(r.groupby("arch").accuracy.agg(["count","mean","std"]).round(4).to_string())

In [ ]:
pi = pd.read_csv("/content/drive/MyDrive/classification_task/repro/xai/xai_per_image.csv")
n = pi.groupby(["arch","role","true_class"]).spearman.apply(lambda s: s.isna().mean())
print(f"max {n.max():.3f} chez {n.idxmax()} | min {n.min():.3f}")

In [ ]:
print(pi.groupby("true_class").background_fraction.mean().round(4).to_string())

In [ ]:
pi = pd.read_csv("/content/drive/MyDrive/classification_task/repro/xai/xai_per_image.csv")
print(pi[pi.role=="deep"].groupby("true_class").lesion_area.mean().round(4).to_string())
print()
ctl = pd.read_csv("/content/drive/MyDrive/classification_task/repro/xai/instrument_controls.csv")
print(ctl.to_string(index=False))

In [ ]:
%cd /content/brisc-xai
from google.colab import files
files.upload()      # 05_gradcam_target_2x2.py
!python 05_gradcam_target_2x2.py

In [ ]:
import pandas as pd
d = pd.read_csv("/content/drive/MyDrive/classification_task/repro/xai/gradcam_2x2.csv")
g = d[(d.target=="logit") & (d.true_class=="meningioma") & d.degenerate]
print("par architecture :"); print(g.groupby("arch").size().to_string())
print("\nper seed :"); print(g.groupby(["arch","seed"]).size().to_string())
print(f"\nconfidence of degenerate cases : {g.confidence.mean():.4f} "
      f"versus {d[(d.target=='logit') & (d.true_class=='meningioma')].confidence.mean():.4f} on average")

In [ ]:
d2 = d[~((d.arch=="CNN_GAP") & (d.seed==1337))]
print(d2.pivot_table(index=["true_class","policy"], columns="target", values="degenerate").round(4).to_string())

In [ ]:
%cd /content/brisc-xai
!rm -f 03_measure_xai.py 04_make_tables.py
from google.colab import files
files.upload()
!python 03_measure_xai.py
!python 04_make_tables.py

In [ ]:
%cd /content/brisc-xai
!mv "03_measure_xai (1).py" 03_measure_xai.py
!grep -c "permuted_null_within_class" 03_measure_xai.py
!python 03_measure_xai.py

In [ ]:
import pandas as pd, os
t4 = pd.read_csv(os.path.join(os.environ["BRISC_OUT"], "tables", "t4_alignment.csv"))
print(t4[t4.role=="deep"][["arch","true_class","dice","null_dice","dice_minus_null","uniform_floor","ceiling"]].to_string(index=False))

In [ ]:
!python 04_make_tables.py

In [ ]:
print(t4[t4.role=="deep"][["arch","true_class","heatmap_area","null_dice"]].to_string(index=False))

In [ ]:
os.environ["BRISC_ROOT"] = "/content/brisc_fresh/brisc2025"
os.environ["BRISC_OUT"]  = "/content/drive/MyDrive/classification_task/repro"

In [ ]:
!cp -r /content/drive/MyDrive/brisc-xai /content/

In [ ]:
%cd /content/brisc-xai
!mv "03_measure_xai (2).py" 03_measure_xai.py
!mv "04_make_tables (1).py" 04_make_tables.py
!grep -c "auc_upsampled" 03_measure_xai.py 04_make_tables.py
!python 03_measure_xai.py

In [ ]:
%cd /content/brisc-xai
!sed -i 's/    epochs=120,/    epochs=200,/' config.py
!sed -i 's/    seeds=\[42, 123, 2024\],/    seeds=[42, 123, 2024, 7, 1337],/' config.py
!sed -i 's/version="frozen-2026-09-10"/version="frozen-2026-09-13-5seeds"/' config.py
!cp config.py /content/drive/MyDrive/brisc-xai/config.py
!python -c "import config as C; print(C.PROTOCOL_HASH)"

In [ ]:
!python 03_measure_xai.py

In [ ]:
import pandas as pd, os
t4 = pd.read_csv(os.path.join(os.environ["BRISC_OUT"], "tables", "t4_alignment.csv"))
print(t4[t4.role=="deep"][["arch","true_class","dice","null_dice","dice_minus_null","uniform_floor","ceiling"]].to_string(index=False))
print()
print(t4[t4.role=="deep"][["arch","true_class","auc_intrabrain","auc_upsampled","heatmap_area"]].to_string(index=False))

In [ ]:
%cd /content/brisc-xai
!grep -c "auc_upsampled" 03_measure_xai.py
!ls -la 03_measure_xai.py

In [ ]:
!python 03_measure_xai.py
!python 04_make_tables.py

In [ ]:
import pandas as pd, os
t4 = pd.read_csv(os.path.join(os.environ["BRISC_OUT"], "tables", "t4_alignment.csv"))
print(t4[t4.role=="deep"][["arch","true_class","auc_intrabrain","auc_upsampled","heatmap_area"]].to_string(index=False))

In [ ]:
%cd /content/brisc-xai
!mv "03_measure_xai (3).py" 03_measure_xai.py
!mv "04_make_tables (2).py" 04_make_tables.py
!grep -c "auc_upsampled_ib" 03_measure_xai.py
!python 03_measure_xai.py

In [ ]:
!python 04_make_tables.py


In [ ]:
import pandas as pd
t = pd.read_csv("/content/drive/MyDrive/classification_task/repro/xai/xai_per_image.csv")
d = t[t.role=="deep"].groupby(["arch","true_class"]).agg(
    full=("auc_full","mean"), up=("auc_upsampled","mean"),
    ib=("auc_intrabrain","mean"), up_ib=("auc_upsampled_ib","mean"),
    area=("heatmap_area","mean")).round(4)
d["infl_full"] = (d.up - d.full).round(4)
d["infl_ib"] = (d.up_ib - d.ib).round(4)
print(d.to_string())

In [ ]:
import os
os.environ["BRISC_OUT"] = "/content/drive/MyDrive/classification_task/repro"
!cp -r /content/drive/MyDrive/brisc-xai /content/
%cd /content/brisc-xai

In [ ]:
import os, getpass
!pip install -q kaggle
if not os.path.isdir("/content/brisc_fresh/brisc2025"):
    tok = getpass.getpass("token kaggle : ")
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    open(os.path.expanduser("~/.kaggle/access_token"), "w").write(tok.strip())
    os.chmod(os.path.expanduser("~/.kaggle/access_token"), 0o600)
    del tok
    !kaggle datasets download -d briscdataset/brisc2025 -p /content/brisc_fresh --unzip
os.environ["BRISC_ROOT"] = "/content/brisc_fresh/brisc2025"
!python 06_make_figures.py

In [ ]:
%cd /content/brisc-xai
from google.colab import files
files.upload()      # 06_make_figures.py
!ls -la 06*

In [ ]:
!pip install -q matplotlib
!python 06_make_figures.py

In [ ]:
!cp -r /content/brisc-xai/. /content/drive/MyDrive/brisc-xai/

In [ ]:
%cd /content/brisc-xai
!rm -rf __pycache__ *\(*\)*
!python -c "import config as C; print(C.PROTOCOL_HASH)"
!ls

In [ ]:
!ls -la /content/drive/MyDrive/classification_task/repro/xai/figures

In [ ]:
import os

# Check whether Google Drive is mounted
print("Drive mounted:", os.path.isdir("/content/drive"))

# Look for folders/files that look like the BRISC dataset
for root, dirs, files in os.walk("/content"):
    # limit the depth so as not to flood the output
    depth = root.count(os.sep) - "/content".count(os.sep)
    if depth > 3:
        dirs[:] = []
        continue
    for name in dirs:
        if any(k in name.lower() for k in ["brisc", "train", "test", "val", "glioma", "meningioma", "pituitary"]):
            print("DIR :", os.path.join(root, name))
    for name in files:
        if "manifest" in name.lower() or name.lower().endswith(".csv"):
            print("FILE:", os.path.join(root, name))

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    if "/drive" in root or "/sample_data" in root:
        dirs[:] = []
        continue
    depth = root.count(os.sep) - "/content".count(os.sep)
    if depth > 4:
        dirs[:] = []
        continue
    for name in dirs:
        if any(k in name.lower() for k in ["brisc", "train", "test", "val", "glioma", "meningioma", "pituitary"]):
            print("DIR :", os.path.join(root, name))
    for name in files:
        if "manifest" in name.lower():
            print("FILE:", os.path.join(root, name))

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    if "/drive" in root or "/sample_data" in root:
        dirs[:] = []
        continue
    depth = root.count(os.sep) - "/content".count(os.sep)
    if depth > 4:
        dirs[:] = []
        continue
    for name in dirs:
        if any(k in name.lower() for k in ["brisc", "train", "test", "val", "glioma", "meningioma", "pituitary"]):
            print("DIR :", os.path.join(root, name))
    for name in files:
        if "manifest" in name.lower():
            print("FILE:", os.path.join(root, name))

In [ ]:
import os

base = "/content/brisc_fresh/brisc2025/classification_task"
for split in ["train", "test"]:
    split_path = os.path.join(base, split)
    total = 0
    for cls in sorted(os.listdir(split_path)):
        cls_path = os.path.join(split_path, cls)
        if os.path.isdir(cls_path):
            n = len(os.listdir(cls_path))
            total += n
            print(split, cls, n)
    print(split, "TOTAL", total)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json, os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linewidth": 0.5,
})

P = "/content/drive/MyDrive/classification_task/repro"
HIST_DIR = os.path.join(P, "ablation", "histories")
OUT_DIR = os.path.join(P, "xai", "figures")
os.makedirs(OUT_DIR, exist_ok=True)

ARCHS = [
    ("Baseline", "Baseline_CNN (Flatten)", "#1f4e8c"),
    ("CNN_GAP", "CNN_GAP", "#2e8b3d"),
    ("CNN_GAP_BN", "CNN_GAP_BN", "#c0392b"),
    ("Proposed", "Proposed (CNN_GAP_BN_SE)", "#7d3c98"),
]

data = {name: json.load(open(os.path.join(HIST_DIR, f"{name}__seed42.json")))
        for name, _, _ in ARCHS}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4), dpi=300)

for name, label, color in ARCHS:
    d = data[name]
    ep_tr = range(1, len(d["accuracy"]) + 1)
    ep_va = range(1, len(d["val_accuracy"]) + 1)
    ax1.plot(ep_tr, d["accuracy"], color=color, lw=1.8, ls="-")
    ax1.plot(ep_va, d["val_accuracy"], color=color, lw=1.1, ls="--", alpha=0.85)
    ax2.plot(ep_tr, d["loss"], color=color, lw=1.8, ls="-", label=label)
    ax2.plot(ep_va, d["val_loss"], color=color, lw=1.1, ls="--", alpha=0.85)

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.set_title("(a) Training (solid) and validation (dashed) accuracy", fontsize=10)

ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.set_title("(b) Training (solid) and validation (dashed) loss", fontsize=10)

arch_handles = [Line2D([0], [0], color=c, lw=2.2) for _, _, c in ARCHS]
arch_labels = [lab for _, lab, _ in ARCHS]
fig.legend(arch_handles, arch_labels, loc="lower center", ncol=4,
           bbox_to_anchor=(0.5, -0.06), frameon=False, fontsize=9,
           title="Architecture", title_fontsize=9)

style_handles = [Line2D([0], [0], color="black", lw=1.8, ls="-"),
                  Line2D([0], [0], color="black", lw=1.1, ls="--", alpha=0.85)]
fig.legend(style_handles, ["Training", "Validation"], loc="upper center", ncol=2,
           bbox_to_anchor=(0.5, 1.06), frameon=False, fontsize=9)

fig.tight_layout(rect=[0, 0.06, 1, 0.94])
out = os.path.join(OUT_DIR, "fig1_training_curves.png")
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.close(fig)
print("saved ->", out)

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!unzip -o jeemi_scripts.zip -d /content/brisc-xai
!ls -la /content/brisc-xai/jeemi_scripts

In [ ]:
!find /content/drive/MyDrive/classification_task/repro -iname "*.npy" | head -20
!find /content/drive/MyDrive/classification_task/repro -type d

In [ ]:
import os
print("Drive mounted:", os.path.isdir("/content/drive"))

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls /content/drive/MyDrive/
!find /content/drive/MyDrive -maxdepth 3 -iname "*repro*" -o -iname "*classification_task*"

In [ ]:
!find /content/drive/MyDrive/classification_task -maxdepth 4 -type d
!find /content/drive/MyDrive/classification_task -iname "*.npy" | head -20
!find /content/drive/MyDrive/classification_task -iname "*.csv"

In [ ]:
!find /content/drive/MyDrive/classification_task/repro/xai -type f
!find /content/drive/MyDrive/classification_task/diagnostic_gradcam -type f
!find /content/drive/MyDrive/classification_task -iname "*.npy" | wc -l
!find /content/drive/MyDrive/classification_task -iname "*.npy"

In [ ]:
!grep -n "np.save\|\.npy\|save(" /content/brisc-xai/03_measure_xai.py

In [ ]:
from google.colab import files
files.upload()   # select jeemi_scripts_1.zip

In [ ]:
!unzip -o "jeemi_scripts (1).zip" -d /content/brisc-xai
!cp -r /content/drive/MyDrive/brisc-xai/. /content/brisc-xai/
%cd /content/brisc-xai
!ls config.py models.py
!ls jeemi_scripts/_xai_common.py jeemi_scripts/A6_null_convergence.py

In [ ]:
!cp jeemi_scripts/_xai_common.py jeemi_scripts/A6_null_convergence.py jeemi_scripts/A7_threshold_sensitivity.py /content/brisc-xai/
!ls /content/brisc-xai/*.py

In [ ]:
os.environ["BRISC_ROOT"] = "/content/drive/MyDrive"

!python A6_null_convergence.py --out A6_null_convergence.csv


In [ ]:
!ls "/content/drive/MyDrive/classification_task/test/glioma/brisc2025_test_00001_gl_ax_t1.jpg"

In [ ]:
!mkdir -p /content/brisc_fresh
!ln -s /content/drive/MyDrive /content/brisc_fresh/brisc2025
!ls /content/brisc_fresh/brisc2025/classification_task/test/glioma/brisc2025_test_00001_gl_ax_t1.jpg

In [ ]:
!python A6_null_convergence.py --out A6_null_convergence.csv

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/brisc-xai
!cp -r /content/drive/MyDrive/brisc-xai/. /content/brisc-xai/
!mkdir -p /content/brisc_fresh
!ln -sfn /content/drive/MyDrive /content/brisc_fresh/brisc2025

os.environ["BRISC_OUT"] = "/content/drive/MyDrive/classification_task/repro"
%cd /content/brisc-xai
!ls config.py models.py

In [ ]:
from google.colab import files
files.upload()   # select jeemi_scripts (1).zip

In [ ]:
!unzip -o "jeemi_scripts (1).zip" -d /tmp/js
!cp /tmp/js/jeemi_scripts/*.py /content/brisc-xai/
!cp /tmp/js/jeemi_scripts/*.py /content/drive/MyDrive/brisc-xai/
!ls /content/brisc-xai/A*.py /content/brisc-xai/B3*.py /content/brisc-xai/_xai_common.py

In [ ]:
!nohup python A6_null_convergence.py --out /content/drive/MyDrive/classification_task/repro/A6_null_convergence.csv > /content/drive/MyDrive/classification_task/repro/A6_log.txt 2>&1 &

In [ ]:
!grep -E "XAI set|done|ecrit|Error|Traceback" /content/drive/MyDrive/classification_task/repro/A6_log.txt

In [ ]:
!ps aux | grep -v grep | grep A6_null
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv
!tail -n 15 /content/drive/MyDrive/classification_task/repro/A6_log.txt
!ls -la /content/drive/MyDrive/classification_task/repro/A6_null_convergence.csv

In [ ]:
%cd /content/brisc-xai
!nohup python -u A6_null_convergence.py --out /content/drive/MyDrive/classification_task/repro/A6_null_convergence_fast.csv > /content/drive/MyDrive/classification_task/repro/A6_fast_log.txt 2>&1 &

In [ ]:
!tail -n 5 /content/drive/MyDrive/classification_task/repro/A6_fast_log.txt

In [ ]:
!tail -n 3 /content/drive/MyDrive/classification_task/repro/A6_fast_log.txt

In [ ]:
!pkill -f A6_null_convergence

In [ ]:
!grep -n "dice_matrix" /content/brisc-xai/A6_null_convergence.py
!ls -la /content/*.py /content/brisc-xai/*.py

In [ ]:
!pkill -f A6_null_convergence

In [ ]:
%cd /content
from google.colab import files
files.upload()

In [ ]:
!pkill -f A6_null_convergence
!sleep 2; ps aux | grep A6_null | grep -v grep | wc -l

In [ ]:
!cp /content/A6_null_convergence.py /content/brisc-xai/A6_null_convergence.py
!grep -c "dice_matrix" /content/brisc-xai/A6_null_convergence.py
!ls -la /content/brisc-xai/A6_null_convergence.py

In [ ]:
%cd /content/brisc-xai
!nohup python -u A6_null_convergence.py --out /content/drive/MyDrive/classification_task/repro/A6_null_convergence_fast.csv > /content/drive/MyDrive/classification_task/repro/A6_fast_log.txt 2>&1 &

In [ ]:
!tail -n 3 /content/drive/MyDrive/classification_task/repro/A6_fast_log.txt

In [ ]:
!pkill -f A6_null_convergence

In [ ]:
import pandas as pd
d = pd.read_csv("/content/drive/MyDrive/classification_task/repro/A6_null_convergence_fast.csv")
print(len(d), "rows")
print(d.pivot_table(index=["arch","role"], columns="R", values="null_mean").round(4))
print()
print(d.groupby("R")[["delta_vs_R5"]].agg(["mean","min","max"]).round(4))
print()
print((d.ci95_hi - d.ci95_lo).groupby(d.R).mean().round(4))

In [ ]:
%cd /content/brisc-xai
!nohup python -u A7_threshold_sensitivity.py --out /content/drive/MyDrive/classification_task/repro/A7_threshold.csv > /content/drive/MyDrive/classification_task/repro/A7_log.txt 2>&1 &

In [ ]:
!tail -n 3 /content/drive/MyDrive/classification_task/repro/A7_log.txt

In [ ]:
import pandas as pd
d = pd.read_csv("/content/drive/MyDrive/classification_task/repro/A7_threshold.csv")
print(d.shape, "| metrics:", sorted(d.metric.unique()))
p = d.pivot_table(index=["arch","class"], columns="metric", values="value").round(3)
print(p.to_string())
print()
print(d.groupby("arch")["n_degenerate"].max())

In [ ]:
!sed -n 1,25p /content/brisc-xai/A2_near_duplicate_audit.py

In [ ]:
import os, glob, numpy as np, pandas as pd
os.environ.setdefault("BRISC_OUT","/content/drive/MyDrive/classification_task/repro")
import sys; sys.path.insert(0,"/content/brisc-xai")
import config as C
print("DATASET_DIR:", C.DATASET_DIR)
m = glob.glob(str(C.DATASET_DIR)+"/**/manifest.csv", recursive=True) + glob.glob("/content/drive/MyDrive/classification_task/repro/**/manifest.csv", recursive=True)
print("manifest:", m[:3])
if m:
    mf = pd.read_csv(m[0]); print(mf.columns.tolist()); print(mf.head(3).to_string())
    for c in mf.columns:
        if mf[c].nunique() < 8: print(c, mf[c].value_counts().to_dict())
preds = glob.glob("/content/drive/MyDrive/classification_task/repro/**/*pred*", recursive=True)
print(len(preds), "pred files"); print("\n".join(preds[:6]))
for p in preds[:2]:
    if p.endswith(".npy"):
        a = np.load(p, allow_pickle=True); print(os.path.basename(p), a.shape, a.dtype, a[:3])
try:
    import imagehash; print("imagehash OK")
except Exception as e: print("imagehash absent:", e)

In [ ]:
!pip -q install imagehash

In [ ]:
import os, glob, numpy as np, pandas as pd, imagehash
from PIL import Image
R = "/content/drive/MyDrive/classification_task/repro"
mf = pd.read_csv(R+"/dataset/manifest.csv")
tr = mf[mf.split=="train"].reset_index(drop=True)
te = mf[mf.split=="test"].reset_index(drop=True)

def hx(p): return str(imagehash.phash(Image.open(p).convert("L"), hash_size=8))
for name, df in (("train",tr),("test",te)):
    out = pd.DataFrame({"path":df.path, "class":df.label, "phash_hex":[hx(p) for p in df.path]})
    out.to_csv(f"{R}/A2_phash_{name}.csv", index=False); print(name, len(out), "hashed")
    if name=="test": hte = out

# predictions -> CSV in the A2 format + order check
rows, chk = [], []
for f in sorted(glob.glob(R+"/ablation/histories/*__seed*_pred.npy")):
    arch, s = os.path.basename(f).replace("_pred.npy","").split("__seed"); s=int(s)
    p = np.load(f)
    ok = (p == te.y.values).astype(int)
    chk.append((arch, s, round(ok.mean()*100,2)))
    rows.append(pd.DataFrame({"path":te.path,"class":te.label,"correct":ok,"model":arch,"seed":s}))
pd.concat(rows).to_csv(R+"/A2_pred.csv", index=False)
print(pd.DataFrame(chk, columns=["arch","seed","acc_%"]).to_string(index=False))
ar = pd.read_csv(R+"/ablation/ablation_runs.csv"); print(ar.columns.tolist()); print(ar.head(4).to_string())

In [ ]:
%cd /content/brisc-xai
!python -u A2_near_duplicate_audit.py --phash-train /content/drive/MyDrive/classification_task/repro/A2_phash_train.csv --phash-test /content/drive/MyDrive/classification_task/repro/A2_phash_test.csv --pred /content/drive/MyDrive/classification_task/repro/A2_pred.csv --out /content/drive/MyDrive/classification_task/repro/A2_neardup.csv

In [ ]:
import pandas as pd
R = "/content/drive/MyDrive/classification_task/repro"
d = pd.read_csv(R+"/A2_neardup.csv")
print(d[d.model=="_count"][["threshold","n_near","n_far"]].to_string(index=False))
print()
a = d[(d.model!="_count") & (d.cls=="all")]
print(a.groupby(["threshold","model"])[["n_near","acc_near","acc_far","delta"]].mean().round(3).to_string())
print()
ar = pd.read_csv(R+"/ablation/ablation_runs.csv")
pr = pd.read_csv(R+"/A2_pred.csv").groupby(["model","seed"]).correct.mean().reset_index()
m = pr.merge(ar[["arch","seed","accuracy"]], left_on=["model","seed"], right_on=["arch","seed"])
print("max accuracy difference, recomputed vs ablation_runs:", round(float((m.correct-m.accuracy).abs().max()), 4), "| models compared:", len(m))

In [ ]:
import pandas as pd
R = "/content/drive/MyDrive/classification_task/repro"
mf = pd.read_csv(R+"/dataset/manifest.csv")
tr = mf[mf.split=="train"]; te = mf[mf.split=="test"]
print("test with md5 identical to train/val:", te.md5.isin(set(tr.md5)).sum(), "/", len(te))
print("doublons md5 internes au train :", tr.md5.duplicated().sum())
h = pd.read_csv(R+"/A2_phash_train.csv")
bits = h.phash_hex.map(lambda x: bin(int(x,16))[2:].zfill(64))
print("hash identique au manifest :", (bits.values == tr.reset_index(drop=True).group.values).mean().round(3))

In [ ]:
import numpy as np
g = tr.reset_index(drop=True).group
b = pd.Series(bits.values)
neq = b != g
ham = [bin(int(x,2)^int(y,2)).count("1") for x,y in zip(b[neq], g[neq])]
print("n different:", int(neq.sum())); print(pd.Series(ham).describe().round(1))
print("unique train groups:", g.nunique(), "| max size:", g.value_counts().max())
print("test.group present in train.group:", int(te.group.isin(set(tr.group)).sum()), "| test group NaN :", int(te.group.isna().sum()))
print("groups spanning train/val:", int((tr.groupby("group").split_i.nunique()>1).sum()))
print("duplicated md5 spanning train/val:", int((tr[tr.md5.duplicated(keep=False)].groupby("md5").split_i.nunique()>1).sum()))

In [ ]:
import pandas as pd
R = "/content/drive/MyDrive/classification_task/repro"
ar = pd.read_csv(R+"/ablation/ablation_runs.csv")
x = ar[["arch","seed","accuracy"]].rename(columns={"arch":"architecture"}).copy()
x["accuracy"] = (x.accuracy*100).round(4)
x.to_csv(R+"/A3_input_acc.csv", index=False)
print(x[x.architecture.isin(["Baseline","CNN_GAP"])].groupby("architecture").accuracy.mean().round(2))

In [ ]:
%cd /content/brisc-xai
!python -u A3_equivalence_tost.py --acc /content/drive/MyDrive/classification_task/repro/A3_input_acc.csv --out /content/drive/MyDrive/classification_task/repro/A3_tost.csv

In [ ]:
%cd /content/brisc-xai
!python -u A3_equivalence_tost.py --acc /content/drive/MyDrive/classification_task/repro/A3_input_acc.csv --out /content/drive/MyDrive/classification_task/repro/A3_tost.csv

In [ ]:
import pandas as pd, sys, os
os.environ.setdefault("BRISC_OUT","/content/drive/MyDrive/classification_task/repro")
R = "/content/drive/MyDrive/classification_task/repro"
x = pd.read_csv(R+"/xai/xai_per_image.csv")
print(x.columns.tolist()); print(x.head(2).to_string())
import tensorflow as tf
m = tf.keras.models.load_model(R+"/ablation/models/CNN_GAP__seed1337.keras", compile=False)
for l in m.layers[-6:]: print(l.name, l.__class__.__name__, getattr(l,"activation",None).__name__ if hasattr(l,"activation") else "")
print([l.name for l in m.layers if "conv" in l.name])

In [ ]:
%cd /content
from google.colab import files
files.upload()

In [ ]:
!cp /content/A5_degenerate_diagnosis.py /content/brisc-xai/A5_degenerate_diagnosis.py
!grep -c "def diagnose" /content/brisc-xai/A5_degenerate_diagnosis.py

In [ ]:
%cd /content/brisc-xai
!nohup python -u A5_degenerate_diagnosis.py --out /content/drive/MyDrive/classification_task/repro/A5_per_image.csv --out-summary /content/drive/MyDrive/classification_task/repro/A5_summary.csv > /content/drive/MyDrive/classification_task/repro/A5_log.txt 2>&1 &

In [ ]:
!tail -n 5 /content/drive/MyDrive/classification_task/repro/A5_log.txt

In [ ]:
import pandas as pd
R = "/content/drive/MyDrive/classification_task/repro"
d = pd.read_csv(R+"/A5_per_image.csv")
print(d.groupby(["seed","true_class"]).degenerate.agg(n_degenerate="sum", n="size").to_string())
print()
cols = ["raw_max","frac_raw_pos","frac_alpha_neg","grad_l2","frac_grad_zero","act_mean","act_max","frac_ch_active","logit_cls","p_cls"]
m = d[d.true_class=="meningioma"]
print(m.groupby(["seed","degenerate"])[cols].median().round(4).to_string())
x = pd.read_csv(R+"/xai/xai_per_image.csv")
x = x[(x.arch=="CNN_GAP") & (x.role=="deep")]
print()
print("degenerate in xai_per_image.csv:", int(x.degenerate.sum()), "| in A5:", int(d.degenerate.sum()), "| verify_err max :", d.verify_err.max())

In [ ]:
import pandas as pd
x = pd.read_csv("/content/drive/MyDrive/classification_task/repro/xai/xai_per_image.csv")
g = x[x.arch=="CNN_GAP"]
print(g.groupby(["role","layer"]).degenerate.agg(["sum","size"]))
print(g[(g.true_class=="meningioma") & (g.seed==1337)].groupby(["role","layer"]).degenerate.agg(["sum","size"]))

In [ ]:
import pandas as pd
R = "/content/drive/MyDrive/classification_task/repro"
g = pd.read_csv(R+"/xai/gradcam_2x2.csv"); print(g.columns.tolist()); print(g.head(8).to_string())
d = pd.read_csv(R+"/A5_per_image.csv")
s = d[(d.seed==1337)&(d.true_class=="meningioma")&(d.degenerate)]
print("seed 1337 degenerate: n =", len(s), "| mean confidence =", round(s.p_cls.mean(),3), "| median =", round(s.p_cls.median(),3))

In [ ]:
import pandas as pd
R = "/content/drive/MyDrive/classification_task/repro"
g = pd.read_csv(R+"/xai/gradcam_2x2.csv")
print(g.target.unique(), g.policy.unique(), g.layer.unique())
L = g[(g.target=="logit") & (g.true_class=="meningioma")]
print(L.groupby("policy").degenerate.agg(["sum","size"]))
print(L[(L.arch=="CNN_GAP") & (L.seed==1337)].groupby("policy").degenerate.agg(["sum","size"]))
print("total logit meningioma :", int(L.degenerate.sum()), "/", len(L))

In [ ]:
import os, glob, numpy as np, pandas as pd
R = "/content/drive/MyDrive/classification_task/repro"
mf = pd.read_csv(R+"/dataset/manifest.csv"); te = mf[mf.split=="test"].reset_index(drop=True)
names = {0:"glioma", 1:"meningioma", 2:"no_tumor", 3:"pituitary"}
assert (te.y.map(names) == te.label).all()
rows = []
for f in sorted(glob.glob(R+"/ablation/histories/*__seed*_pred.npy")):
    arch, s = os.path.basename(f).replace("_pred.npy","").split("__seed")
    p = np.load(f)
    rows.append(pd.DataFrame({"image_id":te.stem, "architecture":arch, "seed":int(s),
                              "y_true":te.label, "y_pred":pd.Series(p).map(names)}))
pd.concat(rows).to_csv(R+"/B3_input_pred.csv", index=False)
print(len(pd.concat(rows)), "rows (expected 20000)")
os.makedirs(R+"/B3_conf", exist_ok=True)

In [ ]:
%cd /content/brisc-xai
!python -u B3_perclass_confusion.py --pred /content/drive/MyDrive/classification_task/repro/B3_input_pred.csv --out /content/drive/MyDrive/classification_task/repro/B3_perclass.csv --conf-dir /content/drive/MyDrive/classification_task/repro/B3_conf

In [ ]:
import pandas as pd
b = pd.read_csv("/content/drive/MyDrive/classification_task/repro/B3_perclass.csv")
print(b.round(3).to_string())

In [ ]:
import pandas as pd, glob
for f in sorted(glob.glob("/content/drive/MyDrive/classification_task/repro/B3_conf/*")):
    if "Baseline" in f or "Proposed" in f:
        print(f.split("/")[-1]); print(pd.read_csv(f, index_col=0).to_string()); print()

In [ ]:
!cat /content/brisc-xai/config.py

In [ ]:
import os, subprocess
link = "/content/brisc_fresh/brisc2025"
real = os.path.realpath(link)
print("est un lien :", os.path.islink(link), "| cible :", real)
print(subprocess.run(["du","-sh",real+"/classification_task"],capture_output=True,text=True).stdout)
print(subprocess.run(["df","-h","/content"],capture_output=True,text=True).stdout)

In [ ]:
import os, shutil, time, pandas as pd
from concurrent.futures import ThreadPoolExecutor
R = "/content/drive/MyDrive/classification_task/repro"
mf = pd.read_csv(R+"/dataset/manifest.csv")
paths = list(mf.path) + list(mf.mask_path.dropna())
pre, tmp = "/content/brisc_fresh/brisc2025/", "/content/brisc_local/brisc2025/"
assert all(p.startswith(pre) for p in paths), "chemin inattendu"
def cp(p):
    rel = p[len(pre):]; src = "/content/drive/MyDrive/" + rel; dst = tmp + rel
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst): shutil.copyfile(src, dst)
    return os.path.exists(dst)
t0 = time.time()
with ThreadPoolExecutor(16) as ex: ok = list(ex.map(cp, paths))
print(sum(ok), "/", len(paths), "copied in", round((time.time()-t0)/60,1), "min")

In [ ]:
import os, subprocess
tmp = "/content/brisc_local/brisc2025"
sizes = [os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(tmp) for f in fs]
print("files:", len(sizes), "| empty:", sum(s == 0 for s in sizes), "| total size:", round(sum(sizes)/1e6), "MB")
from PIL import Image
bad = [p for p in list(mf.path) if not (Image.open(p.replace("/content/brisc_fresh/", "/content/brisc_local/")).convert("RGB").size)]
print("images illisibles :", len(bad))

In [ ]:
link = "/content/brisc_fresh/brisc2025"
assert os.path.islink(link)
os.remove(link)                       # remove the LINK only, not Drive
os.rename("/content/brisc_local/brisc2025", link)
print("lien :", os.path.islink(link), "| image lisible :", os.path.exists(mf.path[0]))

In [ ]:
%cd /content
from google.colab import files
files.upload()

In [ ]:
!cp /content/A4_train_se_only.py /content/brisc-xai/A4_train_se_only.py
!grep -c "CNN_GAP_SE" /content/brisc-xai/A4_train_se_only.py

In [ ]:
%cd /content/brisc-xai
!BRISC_OUT=/content/drive/MyDrive/classification_task/repro nohup python -u A4_train_se_only.py > /content/drive/MyDrive/classification_task/repro/A4_log.txt 2>&1 &

In [ ]:
!tail -n 5 /content/drive/MyDrive/classification_task/repro/A4_log.txt

In [ ]:
import pandas as pd
R = "/content/drive/MyDrive/classification_task/repro"
a4 = pd.read_csv(R+"/ablation_A4/ablation_runs.csv")
print(a4.drop(columns=["protocol_hash"]).to_string())
print()
print(open(R+"/A4_log.txt").read().count("val_accuracy"), "epoch lines in total")
import subprocess
print(subprocess.run("grep -n -E 'CNN_GAP_SE__seed|test accuracy|skip' "+R+"/A4_log.txt", shell=True, capture_output=True, text=True).stdout)

In [ ]:
import re, pandas as pd
R = "/content/drive/MyDrive/classification_task/repro"
run, ep, rec = None, 0, {}
for line in open(R+"/A4_log.txt", errors="ignore"):
    m = re.match(r"(CNN_GAP_SE__seed\d+) \|", line)
    if m: run = m.group(1); rec[run] = []; continue
    m = re.match(r"Epoch (\d+)/", line)
    if m: ep = int(m.group(1)); continue
    m = re.search(r"val_accuracy: ([0-9.]+)", line)
    if m and run: rec[run].append((ep, float(m.group(1))))
for r, v in rec.items():
    acc = [a for _, a in v]
    first40 = next((e for e, a in v if a > 0.40), None)
    print(r, "| epochs:", len(v), "| val_acc max: %.3f" % max(acc), "| first epoch >40%%: %s" % first40,
          "| last:", [round(a, 3) for a in acc[-4:]])

In [ ]:
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

In [ ]:
%cd /content
from google.colab import files
files.upload()

In [ ]:
!cp /content/A10_efficiency.py /content/brisc-xai/A10_efficiency.py
!grep -c "def bench" /content/brisc-xai/A10_efficiency.py

In [ ]:
%cd /content/brisc-xai
!BRISC_OUT=/content/drive/MyDrive/classification_task/repro python -u A10_efficiency.py --out /content/drive/MyDrive/classification_task/repro/A10_efficiency.csv --env-out /content/drive/MyDrive/classification_task/repro/A10_env.txt

In [ ]:
import pandas as pd
R = "/content/drive/MyDrive/classification_task/repro"
print(pd.read_csv(R+"/A10_efficiency.csv").to_string())
print(open(R+"/A10_env.txt").read())

In [ ]:
!rm -f /content/A10_efficiency.py
%cd /content
from google.colab import files
files.upload()

In [ ]:
!cp "/content/A10_efficiency (1).py" /content/brisc-xai/A10_efficiency.py
!grep -c "only" /content/brisc-xai/A10_efficiency.py
!ls -la /content/brisc-xai/A10_efficiency.py

In [ ]:
%cd /content/brisc-xai
!BRISC_OUT=/content/drive/MyDrive/classification_task/repro python -u A10_efficiency.py --out /content/drive/MyDrive/classification_task/repro/A10_efficiency.csv --env-out /content/drive/MyDrive/classification_task/repro/A10_env.txt 2>&1 | grep -E "^ecrit|Error|Traceback"

In [ ]:
import pandas as pd
print(pd.read_csv("/content/drive/MyDrive/classification_task/repro/A10_efficiency.csv").to_string())

In [ ]:
%cd /content
from google.colab import files
files.upload()

In [ ]:
!cp "$(ls -t /content/A8_hierarchical_stats*.py | head -1)" /content/brisc-xai/A8_hierarchical_stats.py
!grep -c "boot_mean" /content/brisc-xai/A8_hierarchical_stats.py
%cd /content/brisc-xai
!python -u A8_hierarchical_stats.py --xai /content/drive/MyDrive/classification_task/repro/xai/xai_per_image.csv --out /content/drive/MyDrive/classification_task/repro/A8_hier.csv

In [ ]:
import pandas as pd
a = pd.read_csv("/content/drive/MyDrive/classification_task/repro/A8_hier.csv")
print(a.to_string())

In [ ]:
import pandas as pd, os, glob
R = "/content/drive/MyDrive/classification_task/repro"

x = pd.read_csv(f"{R}/xai/xai_per_image.csv")
print("xai_per_image :", x.shape)
print(list(x.columns))
print(x.head(3).T.to_string())

m = pd.read_csv(f"{R}/dataset/manifest.csv")
print("\nmanifest :", m.shape)
print(list(m.columns))

print("\nmasques :")
t = m[(m.split=="test") & (m.label=="pituitary")].head(3)
print(t[["path","mask_path"]].to_string())

print("\nfichiers pred :", len(glob.glob(f"{R}/ablation/histories/*_pred.npy")))
print(sorted(os.listdir(f"{R}/ablation/histories"))[:4])

In [ ]:
files.upload()

In [ ]:
%%bash
cp "$(ls -t /content/B4_pituitary_gap*.py | head -1)" /content/brisc-xai/B4_pituitary_gap.py
ls -l /content/brisc-xai/B4_pituitary_gap.py
grep -c "ratio_les_nonles" /content/brisc-xai/B4_pituitary_gap.py

In [ ]:
R = "/content/drive/MyDrive/classification_task/repro"
!cd /content/brisc-xai && python -u B4_pituitary_gap.py --xai {R}/xai/xai_per_image.csv --pred {R}/B3_input_pred.csv --out {R}/B4

In [ ]:
import pandas as pd
for f in ["correct_vs_wrong","size","resolution","context"]:
    print("\n=====",f); print(pd.read_csv(f"{R}/B4/B4_{f}.csv").to_string())

In [ ]:
import pandas as pd
x = pd.read_csv(f"{R}/xai/xai_per_image.csv")
h = x[(x.true_class=="pituitary") & (x.role=="head")].copy()
h["nan"] = h.auc_intrabrain.isna()
print(h.groupby("arch").nan.mean().round(3))
print(h.groupby("stem").nan.mean().round(1).value_counts().sort_index())   # same images for each seed?
print(h.groupby("nan")[["heatmap_area","lesion_area","cam_mean_lesion"]].median())

In [ ]:
a = h[h.nan].lesion_area; b = h[~h.nan].lesion_area
print("NaN    : max lesion %.4f | median %.4f" % (a.max(), a.median()))
print("non-NaN: min lesion %.4f | median %.4f" % (b.min(), b.median()))
print("seuil 1/196 = %.4f" % (1/196))
print("NaN images with lesion > 1/196:", (h[h.nan].groupby('stem').lesion_area.first() > 1/196).sum())

In [ ]:
import cv2, numpy as np, pandas as pd
R = "/content/drive/MyDrive/classification_task/repro"
m = pd.read_csv(f"{R}/dataset/manifest.csv")
m = m[(m.split=="test") & (m.label=="pituitary")].reset_index(drop=True)

def has_cell(path, g):
    k = cv2.resize(cv2.imread(path, cv2.IMREAD_GRAYSCALE), (224,224), interpolation=cv2.INTER_NEAREST)/255.0 >= 0.5
    return bool((cv2.resize(k.astype("float32"), (g,g), interpolation=cv2.INTER_AREA) > 0.5).any())

m["cell14"] = [has_cell(p,14) for p in m.mask_path]
m["cell28"] = [has_cell(p,28) for p in m.mask_path]

x = pd.read_csv(f"{R}/xai/xai_per_image.csv")
x = x[x.true_class=="pituitary"]
nan14 = x[x.role=="head"].groupby("stem").cam_mean_lesion.apply(lambda s: s.isna().all())
nan28 = x[x.role=="deep"].groupby("stem").cam_mean_lesion.apply(lambda s: s.isna().all())
m = m.set_index("stem").join(nan14.rename("nan14")).join(nan28.rename("nan28"))
print(pd.crosstab(~m.cell14, m.nan14, rownames=["recalcul : aucune cellule 14x14"], colnames=["CSV : NaN head"]))
print(pd.crosstab(~m.cell28, m.nan28, rownames=["recalcul : aucune cellule 28x28"], colnames=["CSV : NaN deep"]))

In [ ]:
files.upload()

In [ ]:
%%bash
cp "$(ls -t /content/B4b_saliency_location*.py | head -1)" /content/brisc-xai/B4b_saliency_location.py
ls -l /content/brisc-xai/B4b_saliency_location.py
grep -c "def describe" /content/brisc-xai/B4b_saliency_location.py

In [ ]:
R = "/content/drive/MyDrive/classification_task/repro"
!cd /content/brisc-xai && BRISC_OUT={R} nohup python -u B4b_saliency_location.py --out {R}/B4 > /content/B4b_log.txt 2>&1 &

In [ ]:
!tail -5 /content/B4b_log.txt

In [ ]:
import pandas as pd
R = "/content/drive/MyDrive/classification_task/repro"
s = pd.read_csv(f"{R}/B4/B4b_summary.csv")
print(s.set_index(["arch","role"]).T.to_string())

In [ ]:
from IPython.display import Image, display
R = "/content/drive/MyDrive/classification_task/repro"
display(Image(f"{R}/B4/B4b_mean_maps.png"))

In [ ]:
import numpy as np, matplotlib.pyplot as plt
R = "/content/drive/MyDrive/classification_task/repro"
z = np.load(f"{R}/B4/B4b_mean_maps.npz")
order = ["Baseline__deep","CNN_GAP__deep","CNN_GAP_BN__deep","Proposed__deep"]
fig, ax = plt.subplots(1, 5, figsize=(17, 3.9))
ax[0].imshow(z["lesion_freq"], cmap="magma"); ax[0].set_title("Lesion frequency\n(300 pituitary images)", fontsize=10)
for j, k in enumerate(order, 1):
    m = z[k]; im = ax[j].imshow(m, cmap="jet", vmin=0, vmax=np.percentile(m, 99.5))
    ax[j].contour(z["lesion_freq"], levels=[0.1], colors="w", linewidths=1)
    ax[j].set_title(k.replace("__deep"," (deep)"), fontsize=10); plt.colorbar(im, ax=ax[j], fraction=0.046)
for a in ax: a.axis("off")
plt.tight_layout(); plt.savefig(f"{R}/B4/B4b_mean_maps_v2.png", dpi=170); plt.show()

In [ ]:
from google.colab import files
up = files.upload()

In [ ]:
!ls -l /content/B4c_peak_tests*.py

In [ ]:
%%bash
pwd
ls -l /content/brisc-xai/B4c_peak_tests*.py
grep -c "def brain_pixels" /content/brisc-xai/B4c_peak_tests.py

In [ ]:
R = "/content/drive/MyDrive/classification_task/repro"
!cd /content/brisc-xai && python -u B4c_peak_tests.py --b4b {R}/B4/B4b_per_image.csv --manifest {R}/dataset/manifest.csv --out {R}/B4/B4c_summary.csv

In [ ]:
from google.colab import files
up = files.upload()

In [ ]:
%%bash
ls -l /content/brisc-xai/B4c_peak_tests_v3.py
grep -c "p_closer_than_Bboot" /content/brisc-xai/B4c_peak_tests_v3.py

In [ ]:
R = "/content/drive/MyDrive/classification_task/repro"
!cd /content/brisc-xai && python -u B4c_peak_tests_v3.py --b4b {R}/B4/B4b_per_image.csv --manifest {R}/dataset/manifest.csv --out {R}/B4/B4c_summary_v3.csv > /content/B4c_log.txt 2>&1
!grep -E "dist_med_obs|dist_med_boot|frac_lateral_obs|frac_lateral_boot|frac_edge_obs|frac_edge_boot|_Bboot|ecrit|Error" /content/B4c_log.txt

In [ ]:
from google.colab import files
up = files.upload()

In [ ]:
%%bash
ls -l /content/brisc-xai/B4d_argmax_control_v1.py
grep -c "dist_argmax_vs_centroid" /content/brisc-xai/B4d_argmax_control_v1.py

In [ ]:
R = "/content/drive/MyDrive/classification_task/repro"
!cd /content/brisc-xai && BRISC_OUT={R} nohup python -u B4d_argmax_control_v1.py --out {R}/B4 > /content/B4d_log.txt 2>&1 &

In [ ]:
!tail -3 /content/B4d_log.txt

In [ ]:
import pandas as pd
R = "/content/drive/MyDrive/classification_task/repro"
print(pd.read_csv(f"{R}/B4/B4d_summary.csv").set_index(["arch","role"]).T.to_string())

In [ ]:
!cd /content/brisc-xai && python -u B4c_peak_tests_v3.py --b4b {R}/B4/B4d_per_image.csv --manifest {R}/dataset/manifest.csv --out {R}/B4/B4c_summary_argmax.csv > /content/B4c_argmax_log.txt 2>&1
!grep -E "dist_med_obs|dist_med_boot|frac_lateral_obs|frac_lateral_boot|frac_edge_obs|frac_edge_boot|_Bboot|ecrit|Error" /content/B4c_argmax_log.txt

In [ ]:
c = pd.read_csv(f"{R}/B4/B4c_summary_argmax.csv").set_index("arch")
print(c[["dist_med_obs","dist_med_A_mean","dist_med_A_lo","dist_med_A_hi","p_closer_than_A"]])

In [ ]:
from google.colab import files
up = files.upload()

In [ ]:
R = "/content/drive/MyDrive/classification_task/repro"
!cd /content/brisc-xai && python -u B7_fig45_v1.py --xai {R}/xai/xai_per_image.csv --out {R}/B7

In [ ]:
from IPython.display import Image, display
R = "/content/drive/MyDrive/classification_task/repro"
display(Image(f"{R}/B7/Fig4_dice_references.png", width=1100))
display(Image(f"{R}/B7/Fig5_auc_by_seed.png", width=1100))

In [ ]:
from google.colab import files
up = files.upload()

In [ ]:
R = "/content/drive/MyDrive/classification_task/repro"
!cd /content/brisc-xai && python -u B7_fig45_v2.py --xai {R}/xai/xai_per_image.csv --out {R}/B7

In [ ]:
from google.colab import files
up = files.upload()

In [ ]:
R = "/content/drive/MyDrive/classification_task/repro"
!cd /content/brisc-xai && BRISC_OUT={R} python -u B7_fig2_v1.py --xai {R}/xai/xai_per_image.csv --out {R}/B7

In [ ]:
from IPython.display import Image, display
display(Image(f"{R}/B7/Fig2_gradcam_cases.png", width=1300))

In [ ]:
files.upload()

In [ ]:
ls -l B7_fig2_v2.py

In [ ]:
!grep -c "0.8 \* cam" B7_fig2_v2.py

In [ ]:
!cd /content/brisc-xai && BRISC_OUT=$BRISC_OUT python -u B7_fig2_v2.py --xai $BRISC_OUT/xai/xai_per_image.csv --out $BRISC_OUT/B7

In [ ]:
!find /content/drive/MyDrive/classification_task/repro -name "gradcam_2x2*"

In [ ]:
import pandas as pd
R = "/content/drive/MyDrive/classification_task/repro/xai/"
d = pd.read_csv(R + "gradcam_2x2.csv")
s = pd.read_csv(R + "gradcam_2x2_summary.csv")
print("DETAIL", d.shape); print(d.columns.tolist()); print(d.head(12).to_string())
print("\nSUMMARY", s.shape); print(s.to_string())

In [ ]:
   !find /content/drive/MyDrive/classification_task/repro -iname "*pred*.csv"

In [ ]:
import pandas as pd
d = pd.read_csv("/content/drive/MyDrive/classification_task/repro/B3_input_pred.csv")
print(d.shape); print(d.columns.tolist()); print(d.head(3).to_string())
print(d.groupby(["architecture","seed"]).size() if "architecture" in d.columns else "")

In [ ]:
   from google.colab import files
   import os
   os.chdir("/content/brisc-xai")
   up = files.upload()

In [ ]:
   !ls -l /content/brisc-xai/B3_boot_v1.py

In [ ]:
   !cd /content/brisc-xai && python -u B3_boot_v1.py --pred /content/drive/MyDrive/classification_task/repro/B3_input_pred.csv --out-dir /content/drive/MyDrive/classification_task/repro/B3

In [ ]:
   import pandas as pd
   for a in ["Baseline","CNN_GAP","CNN_GAP_BN","Proposed"]:
       print(a); print(pd.read_csv(f"/content/drive/MyDrive/classification_task/repro/B3/B3_confusion_{a}.csv", index_col=0).to_string())

In [ ]:
import pandas as pd
d = pd.read_csv("/content/drive/MyDrive/classification_task/repro/B3_input_pred.csv")
def n(g, t, p): return int(((g.y_true==t)&(g.y_pred==p)).sum())
rows = []
for (a, s), g in d.groupby(["architecture","seed"]):
    rows.append(dict(arch=a, seed=s, men_to_pit=n(g,"meningioma","pituitary"),
                     gli_to_pit=n(g,"glioma","pituitary"), gli_to_men=n(g,"glioma","meningioma"),
                     acc=round((g.y_true==g.y_pred).mean(),3)))
print(pd.DataFrame(rows).sort_values(["arch","seed"]).to_string(index=False))

In [ ]:
  from IPython.display import Image
  Image("/content/drive/MyDrive/classification_task/repro/B7/Fig2_gradcam_cases_v2.png", width=1100)

In [ ]:
files.upload()

In [ ]:
!ls -l /content/brisc-xai/B7_fig2_v3.py

In [ ]:
   !cd /content/brisc-xai && BRISC_OUT=$BRISC_OUT python -u B7_fig2_v3.py --xai $BRISC_OUT/xai/xai_per_image.csv --out $BRISC_OUT/B7

In [ ]:
import pandas as pd
R = "/content/drive/MyDrive/classification_task/repro/xai/"
d = pd.read_csv(R + "gradcam_2x2.csv")
def tab(x):
    t = x.groupby(["true_class","target","policy"]).degenerate.mean().unstack(["target","policy"]).round(4)
    return t
print("20 RUNS"); print(tab(d))
print("19 RUNS (without CNN_GAP__seed1337)"); print(tab(d[d.run_id != "CNN_GAP__seed1337"]))

In [ ]:
!find /content/drive/MyDrive/classification_task/repro -iname "*A3*" | head -20

In [ ]:
import pandas as pd
R = "/content/drive/MyDrive/classification_task/repro/"
t = pd.read_csv(R + "A3_tost.csv"); a = pd.read_csv(R + "A3_input_acc.csv")
print("TOST", t.shape); print(t.to_string())
print("\nACC", a.shape); print(a.head(25).to_string())

In [ ]:
!ls /content/drive/MyDrive/classification_task/repro | head -30

In [ ]:
import pandas as pd
R = "/content/drive/MyDrive/classification_task/repro/"
print(pd.read_csv(R + "A10_efficiency.csv").to_string())
print("\n--- ENV ---"); print(open(R + "A10_env.txt").read())

In [ ]:
import pandas as pd
R = "/content/drive/MyDrive/classification_task/repro/"
print(pd.read_csv(R + "A2_neardup.csv").head(30).to_string())
print("\n--- A7_log ---"); print(open(R + "A7_log.txt").read()[-3000:])

In [ ]:
!find /content/drive/MyDrive/classification_task -iname "*SE*seed*" -not -iname "*BN*" | head -50

In [ ]:
!ls /content/drive/MyDrive/classification_task/experiments/

In [ ]:
!find /content/drive/MyDrive/classification_task -type f \( -iname "*GAP_SE*" -o -iname "*SE_only*" -o -iname "*SEonly*" \) | head -30

In [ ]:
!find /content/drive/MyDrive/classification_task -type f -name "*seed*.json" | grep -i "CNN_GAP" | sed 's/__seed.*//' | sort | uniq -c

In [ ]:
!grep -i "BN_SE\|Proposed" /content/drive/MyDrive/classification_task/repro/ablation/ablation_runs.csv | cut -d, -f1-4

In [ ]:
!head -1 /content/drive/MyDrive/classification_task/repro/ablation/ablation_runs.csv
!grep -i "Proposed" /content/drive/MyDrive/classification_task/repro/ablation/ablation_runs.csv

In [ ]:
!grep -n "policy\|mixed\|float16\|BATCH\|32" /content/brisc-xai/A10_efficiency.py

In [ ]:
!find /content /content/drive/MyDrive -name "A10_efficiency*.py" 2>/dev/null

In [ ]:
!grep -n "policy\|mixed\|float16\|bfloat\|BATCH\|batch\|32" /content/drive/MyDrive/brisc-xai/A10_efficiency.py

In [ ]:
!ls -la /content/brisc-xai/ | grep -i "A10"
!head -3 /content/drive/MyDrive/classification_task/repro/A10_efficiency.csv

In [ ]:
!find /content/drive/MyDrive -name "xai_per_image.csv" -o -name "manifest.csv" | head

In [ ]:
!cd /content/brisc-xai && python -u B4_pituitary_v1.py \
  --xai /content/drive/MyDrive/classification_task/repro/xai/xai_per_image.csv \
  --runs /content/drive/MyDrive/classification_task/repro/ablation/ablation_runs.csv \
  --preds /content/drive/MyDrive/classification_task/repro/ablation/histories \
  --manifest /content/drive/MyDrive/classification_task/repro/dataset/manifest.csv \
  --out /content/drive/MyDrive/classification_task/repro/B4

In [ ]:
!python -u /content/B4_pituitary_v1.py \
  --xai /content/drive/MyDrive/classification_task/repro/xai/xai_per_image.csv \
  --runs /content/drive/MyDrive/classification_task/repro/ablation/ablation_runs.csv \
  --preds /content/drive/MyDrive/classification_task/repro/ablation/histories \
  --manifest /content/drive/MyDrive/classification_task/repro/dataset/manifest.csv \
  --out /content/drive/MyDrive/classification_task/repro/B4